# Physics-Informed CNN-LSTM for Speed and Heading-Rate Prediction from LiDAR Road Geometry — revision (v5)

Companion notebook of manuscript **sensors-4700056** (revised version). It reproduces every table and figure of the revised article in a single run.

**What changed with respect to the submitted version (v3)**

| Change | Reviewer comment |
|---|---|
| Banking angle measured **relative to the turn direction** (positive = outer edge higher) | R1-3 |
| Physical limit uses a **road curvature fitted to the LiDAR preview** of the current frame (no vehicle-derived radius, no future information) | R2-4, R2-5 |
| **5-fold sequence-wise cross-validation**: every PandaSet sequence is tested exactly once; per-sequence paired Wilcoxon tests and bootstrap 95% CIs | R2-10, R2-11 |
| **Sensitivity of the physics term** to the friction coefficient μ and the weight λ, with activation rates | R1-1, R1-3, R2-1, R2-2, R2-6, R3 |
| **Ablations**: dynamics only, geometry only, no preview, no banking/grade, single preview distances, history length | R1-2, R2-3, R2-17 |
| **Baselines with the same inputs**: constant-trend extrapolation, ridge regression, MLP, LSTM, GRU | R1-4, R2-16 |
| **Curve-stratified** errors by heading-rate magnitude | R2-14 |
| **Quantitative validation** of the preview offsets and road curvature against the vehicle's own future path | R2-13 |
| **PandaSet vs KITTI distributions** (histograms + Kolmogorov–Smirnov) and more KITTI Road sequences | R1-5, R2-8, R2-9 |
| List of sequences per fold and selection criterion | R1-6, R2-12, R3 |

**v5 (after the first revision run):** the comparison with simple baselines showed that the recurrent networks stopped after 1–3 useful epochs and could not exploit the speed trend that a linear model captures. v5 therefore (i) adds the first differences of speed and heading rate (acceleration, heading-rate change) as explicit input channels for **all** learned models, and (ii) uses a patience of 20 epochs with a learning-rate schedule. All other settings are unchanged; features are reused from the v4 cache, so neither PandaSet nor KITTI is re-read.

Intermediate results are saved to Google Drive after every fold, so if Colab disconnects, simply run all cells again: finished folds are loaded instead of retrained.

## 0. Kaggle credentials (optional)

Only needed if PandaSet is downloaded through the Kaggle API. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab **Secrets** panel first.

In [ ]:
# Configure Kaggle API credentials from Colab Secrets (never stored in the notebook).
import json, os
from google.colab import userdata

kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("Kaggle credentials configured securely (not stored in the notebook).")

## 1. Google Drive and global parameters

In [ ]:
# Mount Google Drive (PandaSet sequences are expected under /content/drive/MyDrive/pandaset/).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Global parameters of the experiment.
import os, glob, json, time, pickle, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd

FRECUENCIA_HZ = 10                      # PandaSet and KITTI raw (synced) are sampled at 10 Hz
WINDOW_SIZE = 20                        # 20 past frames = 2.0 s of history
VENTANAS_ABLACION = [10, 30]            # alternative history lengths (1.0 s and 3.0 s) for the ablation
HORIZONTES = [10, 20, 30]               # prediction horizons: 1.0, 2.0 and 3.0 s
HORIZONTE_PRINCIPAL = 10                # horizon used for ablations, sensitivity and stratification (1.0 s)
DISTANCIAS_PREVIEW = [10, 20, 30, 40]   # look-ahead distances (m) used as network inputs
DISTANCIAS_CURVATURA = [10, 15, 20, 25, 30, 35, 40, 45]   # offsets used only to fit the road curvature
SEMILLA = 42
N_FOLDS = 5                             # sequence-wise cross-validation folds
FRACCION_VAL = 0.15                     # share of the non-test sequences used for early stopping
EPOCAS, PACIENCIA, BATCH = 200, 20, 64   # v5: longer patience + learning-rate schedule

RUTA_PANDASET = "/content/drive/MyDrive/pandaset/"
RUTA_KITTI = "/content/kitti_road/"
RUTA_RESULTADOS = "/content/drive/MyDrive/pidl_revision_v5/"   # results per fold (v5)
RUTA_CACHE_FEATURES = "/content/drive/MyDrive/pidl_revision_v4/features_v4.pkl"   # features extracted by v4 (reused)
FORZAR_EXTRACCION = False               # True = re-read all LiDAR frames even if the cache exists
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

# Network inputs (same definition in PandaSet and KITTI)
FEATURE_COLS = [f"preview_{d}m" for d in DISTANCIAS_PREVIEW] + ["peralte_deg", "pendiente_pct", "velocidad_kmh", "heading_rate"]
N_FEATURES = len(FEATURE_COLS)
IDX_VEL, IDX_HR = FEATURE_COLS.index("velocidad_kmh"), FEATURE_COLS.index("heading_rate")

# Physics-informed term
G_FISICA = 9.81                         # m/s^2
MU_FISICA = 0.7                         # nominal dry-pavement friction coefficient
LAMBDA_FISICA = 0.1                     # nominal weight of the physics penalty
MU_SENSIBILIDAD = [0.7, 0.5, 0.3, 0.2, 0.1]   # dry -> wet -> snow -> ice
LAMBDA_SENSIBILIDAD = [0.1, 1.0, 10.0]
R_MIN, R_MAX = 5.0, 1000.0              # clipping of the road radius (m); R_MAX = straight road

print(f"Features ({N_FEATURES}): {FEATURE_COLS}")
print(f"Window: {WINDOW_SIZE / FRECUENCIA_HZ:.1f} s | horizons: {[h / FRECUENCIA_HZ for h in HORIZONTES]} s")

## 2. LiDAR road-geometry features

All points are expressed in the vehicle frame (x forward, y left, z up). For each frame:

* **Ground points**: band of −0.15/+0.30 m around the 10th height percentile inside 0–50 m ahead and |y| < 15 m.
* **Preview offset** $y_d$: center of the lateral ground extent (10th–90th percentiles of y) in a 5 m slice centered d meters ahead.
* **Road curvature** $\kappa$: quadratic fit $y(x)=c_0+c_1x+c_2x^2$ to the offsets at 10–45 m, $\kappa = 2c_2/(1+(c_1+2c_2x_0)^2)^{3/2}$ at $x_0 = 25$ m (positive = road bends left). The physical radius is $R = 1/|\kappa|$, clipped to 5–1000 m.
* **Banking relative to the turn** $\theta$: height difference between the outer and the inner half of the ground within 20 m, divided by their lateral separation. Positive θ means the outer edge is higher (favorable superelevation) for both left and right curves.
* **Grade**: height change between ground points beyond 40 m and closer than 10 m, over 30 m.

In [ ]:
# Geometry utilities shared by the PandaSet and KITTI pipelines.

MIN_DESPLAZAMIENTO_M = 0.05   # below this displacement per frame (0.5 m/s) the yaw is held constant

def calcular_yaw_trayectoria(x, y, min_desp=MIN_DESPLAZAMIENTO_M):
    # Yaw (rad) per frame from the trajectory: backward difference, held while stopped, unwrapped.
    n = len(x)
    yaw = np.full(n, np.nan)
    ultimo = None
    for k in range(1, n):
        dx, dy = x[k] - x[k - 1], y[k] - y[k - 1]
        if np.hypot(dx, dy) >= min_desp:
            ultimo = np.arctan2(dy, dx)
        if ultimo is not None:
            yaw[k] = ultimo
    validos = ~np.isnan(yaw)
    if not validos.any():
        return np.zeros(n)
    yaw[~validos] = yaw[validos][0]
    return np.unwrap(yaw)


def mundo_a_vehiculo(puntos, origen_xy, yaw):
    # World -> vehicle-aligned frame: p_v = R_z(-yaw) (p_w - t); z unchanged.
    c, s = np.cos(yaw), np.sin(yaw)
    dx = puntos[:, 0] - origen_xy[0]
    dy = puntos[:, 1] - origen_xy[1]
    return np.column_stack([c * dx + s * dy, -s * dx + c * dy, puntos[:, 2]])


def ajustar_curvatura(distancias, offsets, x0=25.0):
    # Signed curvature (1/m, positive = left) of y(x) = c0 + c1 x + c2 x^2 fitted to the preview offsets.
    d, y = np.asarray(distancias, float), np.asarray(offsets, float)
    ok = ~np.isnan(y)
    if ok.sum() < 4:
        return np.nan
    c2, c1, _ = np.polyfit(d[ok], y[ok], 2)
    return 2 * c2 / (1 + (c1 + 2 * c2 * x0) ** 2) ** 1.5


def extraer_features_lidar(p):
    # Road-geometry features of one LiDAR frame in the vehicle frame (NaN when not measurable).
    todas = sorted(set(DISTANCIAS_PREVIEW) | set(DISTANCIAS_CURVATURA))
    salida = {f"preview_{d}m": np.nan for d in todas}
    salida.update({"pendiente_pct": np.nan, "theta_raw_rad": np.nan, "kappa_raw": np.nan})

    roi = p[(p[:, 0] > 0) & (p[:, 0] < 50) & (np.abs(p[:, 1]) < 15)]
    if len(roi) < 50:
        return salida
    z_base = np.percentile(roi[:, 2], 10)
    suelo = roi[(roi[:, 2] > z_base - 0.15) & (roi[:, 2] < z_base + 0.30)]
    if len(suelo) < 50:
        return salida

    for d in todas:
        tramo = suelo[np.abs(suelo[:, 0] - d) < 2.5]
        if len(tramo) >= 10:
            salida[f"preview_{d}m"] = 0.5 * (np.percentile(tramo[:, 1], 10) + np.percentile(tramo[:, 1], 90))
    salida["kappa_raw"] = ajustar_curvatura(DISTANCIAS_CURVATURA, [salida[f"preview_{d}m"] for d in DISTANCIAS_CURVATURA])

    lejos, cerca = suelo[suelo[:, 0] > 40, 2], suelo[suelo[:, 0] < 10, 2]
    if len(lejos) > 0 and len(cerca) > 0:
        salida["pendiente_pct"] = (lejos.mean() - cerca.mean()) / 30 * 100

    # Raw cross-slope: (z right - z left) / lateral separation, right = y below the near-road center.
    cercano = suelo[suelo[:, 0] < 20]
    if len(cercano) >= 20:
        centro = np.median(cercano[:, 1])
        der, izq = cercano[cercano[:, 1] < centro], cercano[cercano[:, 1] > centro]
        if len(der) > 0 and len(izq) > 0:
            run = max(abs(der[:, 1].mean() - izq[:, 1].mean()), 0.5)
            salida["theta_raw_rad"] = np.arctan2(der[:, 2].mean() - izq[:, 2].mean(), run)
    return salida


def calcular_vmax_fisico_kmh(R, theta, mu=MU_FISICA, g=G_FISICA):
    # v_max = sqrt(g R (tan(theta) + mu) / (1 - mu tan(theta))), in km/h.
    tan_theta = np.tan(theta)
    denom = np.clip(1.0 - mu * tan_theta, 0.05, None)
    return np.sqrt(np.clip(g * R * (tan_theta + mu) / denom, 0.0, None)) * 3.6


def completar_dinamica(df):
    # Validity flags, forward fill of missing geometry, turn-relative banking and road radius.
    todas = sorted(set(DISTANCIAS_PREVIEW) | set(DISTANCIAS_CURVATURA))
    for d in todas:
        df[f"valid_{d}m"] = df[f"preview_{d}m"].notna()
    geo = [f"preview_{d}m" for d in todas] + ["pendiente_pct", "theta_raw_rad", "kappa_raw"]
    df[geo] = df[geo].ffill().fillna(0.0)

    # Trailing median over 3 frames (no look-ahead) to remove single-frame outliers of the fit
    df["kappa"] = df["kappa_raw"].rolling(3, min_periods=1).median()
    df["R_road_m"] = np.clip(1.0 / np.maximum(np.abs(df["kappa"].values), 1e-6), R_MIN, R_MAX)
    giro = np.where(df["kappa"].values >= 0, 1.0, -1.0)          # +1 left curve, -1 right curve
    df["theta_rad"] = df["theta_raw_rad"].values * giro           # positive = outer edge higher
    df["peralte_deg"] = np.degrees(df["theta_rad"].values)

    # Vehicle-derived radius R = v / omega (submitted version), kept only for comparison
    omega = np.maximum(np.abs(df["heading_rate"].values) * FRECUENCIA_HZ, 1e-3)
    df["R_veh_m"] = np.clip((df["velocidad_kmh"].values / 3.6) / omega, R_MIN, R_MAX)
    df["vmax_kmh"] = calcular_vmax_fisico_kmh(df["R_road_m"].values, df["theta_rad"].values)
    return df

## 3. PandaSet loader

In [ ]:
# Per-frame features for one PandaSet sequence (points are given in world coordinates).

def cargar_secuencia_pandaset(id_secuencia, ruta_base=RUTA_PANDASET):
    ruta_lidar = os.path.join(ruta_base, id_secuencia, "lidar")
    ruta_poses = os.path.join(ruta_lidar, "poses.json")
    if not os.path.exists(ruta_poses):
        return None
    archivos = sorted(f for f in os.listdir(ruta_lidar) if f.endswith(".pkl"))
    with open(ruta_poses, "r") as f:
        poses = json.load(f)
    n = min(len(archivos), len(poses))
    if n < 2:
        return None

    px = np.array([poses[i]["position"]["x"] for i in range(n)])
    py = np.array([poses[i]["position"]["y"] for i in range(n)])
    pz = np.array([poses[i]["position"]["z"] for i in range(n)])

    desp = np.sqrt(np.diff(px)**2 + np.diff(py)**2 + np.diff(pz)**2)
    velocidad = np.concatenate([[desp[0]], desp]) * FRECUENCIA_HZ * 3.6
    yaw = calcular_yaw_trayectoria(px, py)
    heading_rate = pd.Series(np.diff(yaw, prepend=yaw[0])).rolling(3, min_periods=1).mean().values

    filas = []
    for i in range(n):
        try:
            puntos = pd.read_pickle(os.path.join(ruta_lidar, archivos[i]))[["x", "y", "z"]].values
            feats = extraer_features_lidar(mundo_a_vehiculo(puntos, (px[i], py[i]), yaw[i]))
        except Exception:
            feats = extraer_features_lidar(np.zeros((0, 3)))
        feats.update({"frame": i, "velocidad_kmh": velocidad[i], "heading_rate": heading_rate[i],
                      "pos_x": px[i], "pos_y": py[i], "yaw": yaw[i]})
        filas.append(feats)
    return completar_dinamica(pd.DataFrame(filas))

## 4. KITTI raw (Road category): download and loader

The submitted version used three Road sequences. The revision uses all sequences of the official KITTI raw **Road** category that download correctly (sequences that fail are skipped and reported). Camera images are deleted after extraction to save disk space.

In [ ]:
# Download and extract the KITTI raw Road sequences (synced data + calibration).
import subprocess, zipfile, shutil

SECUENCIAS_KITTI_ROAD = [
    ("2011_09_26", "0015"), ("2011_09_26", "0027"), ("2011_09_26", "0028"), ("2011_09_26", "0029"),
    ("2011_09_26", "0032"), ("2011_09_26", "0052"), ("2011_09_26", "0070"), ("2011_09_26", "0101"),
    ("2011_09_29", "0004"), ("2011_09_30", "0016"), ("2011_10_03", "0042"), ("2011_10_03", "0047"),
]
os.makedirs(RUTA_KITTI, exist_ok=True)

def zip_valido(ruta_zip):
    return os.path.exists(ruta_zip) and os.path.getsize(ruta_zip) > 1024 and zipfile.is_zipfile(ruta_zip)

def descargar_secuencia_kitti(fecha, drive_id):
    base_url = "https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data"
    nombre_sync = f"{fecha}_drive_{drive_id}_sync"
    destino = os.path.join(RUTA_KITTI, fecha)
    carpeta = os.path.join(destino, fecha, nombre_sync)
    if os.path.isdir(os.path.join(carpeta, "velodyne_points")) and os.path.isdir(os.path.join(carpeta, "oxts")):
        return carpeta                                   # already extracted
    os.makedirs(destino, exist_ok=True)
    for zip_name, url in [(f"{fecha}_calib.zip", f"{base_url}/{fecha}_calib.zip"),
                          (f"{nombre_sync}.zip", f"{base_url}/{fecha}_drive_{drive_id}/{nombre_sync}.zip")]:
        ruta_zip = os.path.join(destino, zip_name)
        if not zip_valido(ruta_zip):
            if os.path.exists(ruta_zip):
                os.remove(ruta_zip)
            print(f"Downloading {zip_name} ...")
            subprocess.run(["wget", "-q", "-O", ruta_zip, url], check=True)
            if not zip_valido(ruta_zip):
                raise ValueError(f"{zip_name} is not a valid zip.")
        subprocess.run(["unzip", "-oq", ruta_zip, "-d", destino], check=True)
        if zip_name.endswith("_sync.zip"):
            os.remove(ruta_zip)                          # free disk space
    for cam in glob.glob(os.path.join(carpeta, "image_0*")):
        shutil.rmtree(cam, ignore_errors=True)           # camera images are not used
    return carpeta

rutas_secuencias_kitti = []
if os.path.exists(RUTA_CACHE_FEATURES) and not FORZAR_EXTRACCION:
    SECUENCIAS_KITTI_ROAD = []                           # features already cached by v4: no download needed
    print("Feature cache found: KITTI download skipped.")
for fecha, drive_id in SECUENCIAS_KITTI_ROAD:
    try:
        ruta = descargar_secuencia_kitti(fecha, drive_id)
        assert os.path.isdir(ruta), f"Expected path not found: {ruta}"
        rutas_secuencias_kitti.append(ruta)
        print(f"OK: {ruta}")
    except Exception as e:
        print(f"Skipped {fecha}_drive_{drive_id}: {e}")
print(f"KITTI Road sequences available: {len(rutas_secuencias_kitti)}")

In [ ]:
# Per-frame features for one KITTI sequence (Velodyne LiDAR + OXTS).

OXTS_COLS = [
    "lat", "lon", "alt", "roll", "pitch", "yaw", "vn", "ve", "vf", "vl", "vu",
    "ax", "ay", "az", "af", "al", "au", "wx", "wy", "wz", "wf", "wl", "wu",
    "pos_accuracy", "vel_accuracy", "navstat", "numsats", "posmode", "velmode", "orimode",
]

def oxts_a_xy(lat, lon):
    # Local metric coordinates (m) from GPS with the Mercator projection used by the KITTI devkit.
    er = 6378137.0
    escala = np.cos(np.radians(lat[0]))
    x = escala * er * np.radians(lon)
    y = escala * er * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))
    return x - x[0], y - y[0]

def cargar_secuencia_kitti(ruta_secuencia):
    ruta_oxts = os.path.join(ruta_secuencia, "oxts", "data")
    ruta_velo = os.path.join(ruta_secuencia, "velodyne_points", "data")
    if not os.path.isdir(ruta_oxts) or not os.path.isdir(ruta_velo):
        print(f"Missing oxts or velodyne data in {ruta_secuencia}")
        return None
    archivos_oxts = sorted(glob.glob(os.path.join(ruta_oxts, "*.txt")))
    archivos_velo = sorted(glob.glob(os.path.join(ruta_velo, "*.bin")))
    n = min(len(archivos_oxts), len(archivos_velo))

    oxts = pd.DataFrame([np.loadtxt(a) for a in archivos_oxts[:n]], columns=OXTS_COLS)
    yaw = np.unwrap(oxts["yaw"].values)
    heading_rate = pd.Series(np.diff(yaw, prepend=yaw[0])).rolling(3, min_periods=1).mean().values
    px, py = oxts_a_xy(oxts["lat"].values, oxts["lon"].values)

    filas = []
    for i in range(n):
        puntos = np.fromfile(archivos_velo[i], dtype=np.float32).reshape(-1, 4)[:, :3]
        feats = extraer_features_lidar(puntos.astype(np.float64))
        feats.update({"frame": i, "velocidad_kmh": oxts["vf"].iloc[i] * 3.6, "heading_rate": heading_rate[i],
                      "pos_x": px[i], "pos_y": py[i], "yaw": yaw[i]})
        filas.append(feats)
    return completar_dinamica(pd.DataFrame(filas))

## 5. Feature extraction (cached on Drive)

Selection criterion: every PandaSet sequence with LiDAR frames and `poses.json` and at least `WINDOW_SIZE + max(HORIZONTES) + 1` = 51 frames; every KITTI Road sequence that downloads correctly.

In [ ]:
# Extract per-frame features once for every sequence (slow step) and cache them on Drive.
ruta_cache = RUTA_CACHE_FEATURES
if os.path.exists(ruta_cache) and not FORZAR_EXTRACCION:
    with open(ruta_cache, "rb") as f:
        dfs_pandaset, dfs_kitti, descartadas = pickle.load(f)
    print("Features loaded from the cache.")
else:
    dfs_pandaset, descartadas = {}, []
    for seq in sorted(d for d in os.listdir(RUTA_PANDASET) if os.path.isdir(os.path.join(RUTA_PANDASET, d))):
        df_seq = cargar_secuencia_pandaset(seq)
        if df_seq is not None and len(df_seq) >= WINDOW_SIZE + max(HORIZONTES) + 1:
            dfs_pandaset[seq] = df_seq
        else:
            descartadas.append(seq)
    dfs_kitti = {}
    for ruta in rutas_secuencias_kitti:
        df_seq = cargar_secuencia_kitti(ruta)
        if df_seq is not None and len(df_seq) >= max(VENTANAS_ABLACION + [WINDOW_SIZE]) + max(HORIZONTES) + 1:
            dfs_kitti[os.path.basename(ruta).replace("_sync", "")] = df_seq
    with open(ruta_cache, "wb") as f:
        pickle.dump((dfs_pandaset, dfs_kitti, descartadas), f)

ids_pandaset = sorted(dfs_pandaset)
ids_kitti = sorted(dfs_kitti)
print(f"PandaSet sequences used: {len(ids_pandaset)} (discarded: {descartadas})")
print(f"KITTI sequences used: {len(ids_kitti)}: {ids_kitti}")

## 6. Dataset description and PandaSet vs KITTI distributions

In [ ]:
# Table of sequences (frames, duration, speed, heading rate) and comparison of the input distributions.
from scipy import stats
import matplotlib.pyplot as plt

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

def guardar_figura(fig, nombre):
    fig.savefig(os.path.join(FIG_DIR, f"{nombre}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{nombre}.png"), dpi=300, bbox_inches="tight")
    print(f"Saved: {FIG_DIR}/{nombre}.pdf and .png")

def guardar_tabla(df, nombre):
    df.to_csv(os.path.join(FIG_DIR, f"{nombre}.csv"), index=False)
    print(f"Saved: {FIG_DIR}/{nombre}.csv")

filas = []
for ds, dfs in [("PandaSet", dfs_pandaset), ("KITTI", dfs_kitti)]:
    for s, d in dfs.items():
        filas.append({"Dataset": ds, "Sequence": s, "Frames": len(d), "Duration (s)": len(d) / FRECUENCIA_HZ,
                      "Mean speed (km/h)": d["velocidad_kmh"].mean(),
                      "Max |heading rate| (deg/s)": np.degrees(np.abs(d["heading_rate"])).max() * FRECUENCIA_HZ,
                      "Min road radius (m)": d["R_road_m"].min()})
tabla_seq = pd.DataFrame(filas)
guardar_tabla(tabla_seq.round(2), "table_sequences")
display(tabla_seq.groupby("Dataset")[["Frames", "Duration (s)", "Mean speed (km/h)"]].agg(["count", "sum", "mean"]).round(1))

# Distributions of the inputs (per frame)
P, K = pd.concat(dfs_pandaset.values()), pd.concat(dfs_kitti.values())
variables = [("velocidad_kmh", "Speed (km/h)", 1), ("heading_rate", "|Heading rate| (deg/s)", "hr"),
             ("preview_20m", "Preview offset at 20 m (m)", 1), ("preview_40m", "Preview offset at 40 m (m)", 1),
             ("peralte_deg", "Turn-relative banking (deg)", 1), ("pendiente_pct", "Grade (%)", 1)]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
filas_ks = []
for ax, (col, nombre, f) in zip(axes.ravel(), variables):
    a, b = P[col].values, K[col].values
    if f == "hr":
        a, b = np.degrees(np.abs(a)) * FRECUENCIA_HZ, np.degrees(np.abs(b)) * FRECUENCIA_HZ
    lo, hi = np.percentile(np.concatenate([a, b]), [1, 99])
    bins = np.linspace(lo, hi, 40)
    ax.hist(a, bins=bins, density=True, alpha=0.55, label="PandaSet")
    ax.hist(b, bins=bins, density=True, alpha=0.55, label="KITTI Road")
    ks = stats.ks_2samp(a, b)
    ax.set_title(f"{nombre}\nKS D = {ks.statistic:.2f}", fontsize=10)
    ax.set_ylabel("Density")
    filas_ks.append({"Variable": nombre, "PandaSet mean": a.mean(), "PandaSet std": a.std(),
                     "KITTI mean": b.mean(), "KITTI std": b.std(), "KS D": ks.statistic, "KS p": ks.pvalue})
axes[0, 0].legend()
fig.tight_layout()
guardar_figura(fig, "fig10_distributions")
plt.show()
tabla_ks = pd.DataFrame(filas_ks)
guardar_tabla(tabla_ks.round(4), "table_distributions_ks")
display(tabla_ks.round(3))

## 7. Quantitative validation of the LiDAR road geometry

There is no mapped lane geometry for these sequences, so the LiDAR estimates are compared with an independent measurement: the **vehicle's own future path** (poses / GPS), transformed into the vehicle frame of the current frame. For each look-ahead distance d, the path offset $y^{path}_d$ is the lateral position of the vehicle when it is d meters ahead.

* The absolute offset differs by the lane position of the vehicle, so the **shape** $y_d - y_{10}$ (bending of the road between 10 m and d) is compared as well.
* The LiDAR curvature κ is compared with the curvature of the same quadratic fitted to the future path.
* Only frames where the slice had enough ground points (not forward-filled) are used.

In [ ]:
# Preview offsets and curvature from LiDAR vs. the vehicle's future path.

def offsets_trayectoria(df, distancias):
    # Lateral offset (m) of the future path at each distance ahead, in the vehicle frame of every frame.
    px, py, yaw = df["pos_x"].values, df["pos_y"].values, df["yaw"].values
    salida = np.full((len(df), len(distancias)), np.nan)
    for i in range(len(df)):
        fut = mundo_a_vehiculo(np.column_stack([px[i:], py[i:], np.zeros(len(df) - i)]), (px[i], py[i]), yaw[i])
        x, y = fut[:, 0], fut[:, 1]
        # keep the path only while it advances (monotonic x), stop at the first reversal
        avance = np.maximum.accumulate(x)
        for k, d in enumerate(distancias):
            j = np.argmax(avance >= d)
            if avance[j] >= d and j > 0 and x[j] > x[j - 1]:
                salida[i, k] = y[j - 1] + (y[j] - y[j - 1]) * (d - x[j - 1]) / (x[j] - x[j - 1])
    return salida

filas_val, puntos_val = [], {}
for ds, dfs in [("PandaSet", dfs_pandaset), ("KITTI", dfs_kitti)]:
    lid, tra, val, kap_l, kap_t = [], [], [], [], []
    for s, d in dfs.items():
        t = offsets_trayectoria(d, DISTANCIAS_CURVATURA)
        l = d[[f"preview_{x}m" for x in DISTANCIAS_CURVATURA]].values
        v = d[[f"valid_{x}m" for x in DISTANCIAS_CURVATURA]].values
        lid.append(l); tra.append(t); val.append(v)
        kap_l.append(d["kappa_raw"].values)
        kap_t.append(np.array([ajustar_curvatura(DISTANCIAS_CURVATURA, fila) for fila in t]))
    lid, tra, val = np.vstack(lid), np.vstack(tra), np.vstack(val)
    kap_l, kap_t = np.concatenate(kap_l), np.concatenate(kap_t)
    i10 = DISTANCIAS_CURVATURA.index(10)
    for k, dist in enumerate(DISTANCIAS_CURVATURA):
        if dist not in DISTANCIAS_PREVIEW:
            continue
        ok = val[:, k] & ~np.isnan(tra[:, k])
        okf = ok & val[:, i10] & ~np.isnan(tra[:, i10])
        fila = {"Dataset": ds, "Distance (m)": dist, "N frames": int(ok.sum()),
                "r offset": stats.pearsonr(lid[ok, k], tra[ok, k])[0] if ok.sum() > 2 else np.nan,
                "MAE offset (m)": np.mean(np.abs(lid[ok, k] - tra[ok, k])) if ok.any() else np.nan}
        if dist != 10 and okf.sum() > 2:
            fl, ft = lid[okf, k] - lid[okf, i10], tra[okf, k] - tra[okf, i10]
            fila.update({"r shape": stats.pearsonr(fl, ft)[0], "MAE shape (m)": np.mean(np.abs(fl - ft))})
            puntos_val[(ds, dist)] = (ft, fl)
        filas_val.append(fila)
    okk = ~np.isnan(kap_l) & ~np.isnan(kap_t)
    filas_val.append({"Dataset": ds, "Distance (m)": "curvature", "N frames": int(okk.sum()),
                      "r offset": stats.spearmanr(kap_l[okk], kap_t[okk])[0] if okk.sum() > 2 else np.nan,
                      "MAE offset (m)": np.nan,
                      "r shape": np.nan, "MAE shape (m)": np.nan})
    puntos_val[(ds, "kappa")] = (kap_t[okk], kap_l[okk])

tabla_val = pd.DataFrame(filas_val)
guardar_tabla(tabla_val.round(4), "table_preview_validation")
display(tabla_val.round(3))
print("For the curvature row, 'r offset' is the Spearman correlation between LiDAR and path curvature.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
for ax, clave, nombre in [(axes[0], 20, "Road shape at 20 m: $y_{20}-y_{10}$ (m)"),
                          (axes[1], 40, "Road shape at 40 m: $y_{40}-y_{10}$ (m)"),
                          (axes[2], "kappa", "Curvature $\\kappa$ (1/m)")]:
    clave_ps = ("PandaSet", clave)
    if clave_ps not in puntos_val:
        continue
    real, est = puntos_val[clave_ps]
    lim = np.percentile(np.abs(np.concatenate([real, est])), 99)
    ax.scatter(real, est, s=4, alpha=0.3)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel("From the future vehicle path")
    ax.set_ylabel("From LiDAR (current frame)")
    ax.set_title(nombre, fontsize=10.5)
fig.suptitle("PandaSet: LiDAR road geometry vs. the vehicle's future path", y=1.02)
fig.tight_layout()
guardar_figura(fig, "fig11_preview_validation")
plt.show()

## 8. Samples and 5-fold sequence-wise cross-validation

* A sample is a window of `W` past frames; targets are the mean heading rate (deg/frame) and the mean speed (km/h) over the next `h` frames.
* The networks predict the **change** with respect to the last observed value (residual targets); the persistence baseline predicts zero change.
* The physical limit of each sample is computed **only from the last observed LiDAR frame** (road radius and turn-relative banking of the road ahead), so no future measurement enters the constraint and the same quantity is available in real time.
* PandaSet sequences are split into 5 folds; each fold is used once as the test set, and 15% of the remaining sequences are used for early stopping. KITTI is never used for training.

In [ ]:
# Sliding-window samples and sequence-wise folds.

def construir_muestras(df, horizon, window=WINDOW_SIZE):
    datos = df[FEATURE_COLS].values
    giro = np.degrees(df["heading_rate"].values)
    vel = df["velocidad_kmh"].values
    R, th = df["R_road_m"].values, df["theta_rad"].values
    n = len(df) - window - horizon + 1
    if n <= 0:
        return None
    X = np.stack([datos[i:i + window] for i in range(n)])
    g = np.array([giro[i + window:i + window + horizon].mean() for i in range(n)])
    v = np.array([vel[i + window:i + window + horizon].mean() for i in range(n)])
    ult = np.arange(n) + window - 1                      # index of the last observed frame
    return {"X": X, "g": g, "v": v, "R": R[ult], "th": th[ult]}


def apilar(dfs, ids, horizon, window=WINDOW_SIZE):
    partes, seqs = [], []
    for s in ids:
        m = construir_muestras(dfs[s], horizon, window)
        if m is not None:
            partes.append(m)
            seqs += [s] * len(m["g"])
    salida = {k: np.concatenate([p[k] for p in partes]) for k in partes[0]}
    salida["seq"] = np.array(seqs)
    return salida


rng = np.random.default_rng(SEMILLA)
orden = np.array(ids_pandaset)
rng.shuffle(orden)
FOLDS_TEST = [sorted(f) for f in np.array_split(orden, N_FOLDS)]

def particion_fold(k):
    test = FOLDS_TEST[k]
    resto = np.array(sorted(set(ids_pandaset) - set(test)))
    np.random.default_rng(SEMILLA + k).shuffle(resto)
    n_val = max(1, int(round(FRACCION_VAL * len(resto))))
    return [str(x) for x in sorted(resto[n_val:])], [str(x) for x in sorted(resto[:n_val])], [str(x) for x in test]

filas = []
for k in range(N_FOLDS):
    tr, va, te = particion_fold(k)
    filas.append({"Fold": k + 1, "N train": len(tr), "N validation": len(va), "N test": len(te),
                  "Validation sequences": " ".join(va), "Test sequences": " ".join(te)})
    print(f"Fold {k + 1}: train {len(tr)}, validation {len(va)}, test {len(te)} | test: {[str(x) for x in te]}")
tabla_folds = pd.DataFrame(filas)
guardar_tabla(tabla_folds, "table_folds")

## 9. Models

All learned models receive the same standardized window and predict the standardized change of both outputs (multi-task). The networks share the same head (Dropout 0.3 → Dense 16 → Dropout 0.3 → two linear outputs) and differ only in the temporal encoder:

| Model | Encoder |
|---|---|
| Ridge | linear regression on the flattened window (α tuned on validation) |
| MLP | Flatten → Dense 64 |
| LSTM | LSTM 32 |
| GRU | GRU 32 |
| CNN-LSTM (proposed) | causal Conv1D 32 → Conv1D 16 → LSTM 32 |

The physics term adds $\lambda \cdot \mathrm{mean}(\mathrm{ReLU}(\hat v - v_{max})^2)$ to the speed loss (in standardized units). Non-learned baselines: **persistence** (zero change) and **constant trend** (linear extrapolation of the last 5 frames to the middle of the horizon).

In [ ]:
# Network builders and the physics-informed loss layer.
import keras
from tensorflow.keras import layers, models, Input
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


class PerdidaFisica(layers.Layer):
    # Adds lam * mean(ReLU(pred - limit)^2) to the training loss and returns pred unchanged.
    def __init__(self, lam, **kwargs):
        super().__init__(**kwargs)
        self.lam = lam

    def call(self, inputs):
        pred, limite = inputs
        if self.lam > 0:
            self.add_loss(self.lam * keras.ops.mean(keras.ops.square(keras.ops.relu(pred - limite))))
        return pred


def construir_red(tipo, n_in, window, lam):
    entradas = Input(shape=(window, n_in), name="secuencia_entrada")
    limite = Input(shape=(1,), name="vmax_std_input")
    if tipo == "CNN-LSTM":
        x = layers.Conv1D(32, 3, padding="causal", activation="relu")(entradas)
        x = layers.Conv1D(16, 3, padding="causal", activation="relu")(x)
        x = layers.LSTM(32)(x)
    elif tipo == "LSTM":
        x = layers.LSTM(32)(entradas)
    elif tipo == "GRU":
        x = layers.GRU(32)(entradas)
    elif tipo == "MLP":
        x = layers.Dense(64, activation="relu")(layers.Flatten()(entradas))
    else:
        raise ValueError(tipo)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    salida_giro = layers.Dense(1, name="out_giro")(x)
    salida_vel = PerdidaFisica(lam, name="out_velocidad")([layers.Dense(1, name="out_velocidad_raw")(x), limite])
    modelo = models.Model([entradas, limite], [salida_giro, salida_vel])
    modelo.compile(optimizer="adam", loss={"out_giro": "mse", "out_velocidad": "mse"})
    return modelo

construir_red("CNN-LSTM", N_FEATURES + 2, WINDOW_SIZE, LAMBDA_FISICA).summary()

## 10. Experiment runner (resumable)

In [ ]:
# Training / prediction of one model specification on one fold.

def ultimos(X):
    # Last observed heading rate (deg/frame) and speed (km/h) of each window.
    return np.degrees(X[:, -1, IDX_HR]), X[:, -1, IDX_VEL]


def tendencia(X, h, n=5):
    # Constant-trend extrapolation of the last n frames to the middle of the horizon.
    t = np.arange(n)
    pend = lambda y: np.polyfit(t, y.T, 1)[0]
    g, v = np.degrees(X[:, -n:, IDX_HR]), X[:, -n:, IDX_VEL]
    paso = (n - 1) + (h + 1) / 2
    ug, uv = ultimos(X)
    return ug + pend(g) * (paso - (n - 1)), uv + pend(v) * (paso - (n - 1))


def ejecutar_modelo(spec, D, h, semilla):
    # spec: dict(name, kind, cols, window, lam, mu). D: dict of sample sets (train/val/test/kitti).
    # Returns predictions per set and training statistics.
    tipo, cols = spec["kind"], spec.get("cols", list(range(N_FEATURES)))
    lam, mu = spec.get("lam", 0.0), spec.get("mu", MU_FISICA)
    tr = D["train"]
    if tipo == "Persistence":
        return {k: ultimos(D[k]["X"]) for k in ("test", "kitti")}, {}
    if tipo == "Trend":
        return {k: tendencia(D[k]["X"], h) for k in ("test", "kitti")}, {}

    sx = StandardScaler().fit(tr["X"].reshape(-1, N_FEATURES))
    # v5: first differences of speed and heading rate (acceleration and heading-rate change per frame)
    # are added as explicit input channels whenever the dynamics are part of the input
    con_dif = IDX_VEL in cols or IDX_HR in cols
    diferencias = lambda X: np.diff(X[:, :, [IDX_VEL, IDX_HR]], axis=1, prepend=X[:, :1, [IDX_VEL, IDX_HR]])
    sd = StandardScaler().fit(diferencias(tr["X"]).reshape(-1, 2))
    ug_tr, uv_tr = ultimos(tr["X"])
    sg = StandardScaler().fit((tr["g"] - ug_tr).reshape(-1, 1))
    sv = StandardScaler().fit((tr["v"] - uv_tr).reshape(-1, 1))

    def entradas(S):
        Xs = sx.transform(S["X"].reshape(-1, N_FEATURES)).reshape(S["X"].shape)[:, :, cols]
        if con_dif:
            dX = diferencias(S["X"])
            Xs = np.concatenate([Xs, sd.transform(dX.reshape(-1, 2)).reshape(dX.shape)], axis=2)
        _, uv = ultimos(S["X"])
        vmax = calcular_vmax_fisico_kmh(S["R"], S["th"], mu=mu)
        return {"secuencia_entrada": Xs, "vmax_std_input": sv.transform((vmax - uv).reshape(-1, 1))}

    def objetivos(S):
        ug, uv = ultimos(S["X"])
        return {"out_giro": sg.transform((S["g"] - ug).reshape(-1, 1)).ravel(),
                "out_velocidad": sv.transform((S["v"] - uv).reshape(-1, 1)).ravel()}

    def deshacer(S, pg, pv):
        ug, uv = ultimos(S["X"])
        return ug + sg.inverse_transform(pg.reshape(-1, 1)).ravel(), uv + sv.inverse_transform(pv.reshape(-1, 1)).ravel()

    info = {}
    t0 = time.time()
    if tipo == "Ridge":
        plano = lambda S: entradas(S)["secuencia_entrada"].reshape(len(S["g"]), -1)
        Y = lambda S: np.column_stack(list(objetivos(S).values()))
        mejor = min([0.1, 1.0, 10.0, 100.0, 1000.0],
                    key=lambda a: mean_squared_error(Y(D["val"]), Ridge(alpha=a).fit(plano(tr), Y(tr)).predict(plano(D["val"]))))
        reg = Ridge(alpha=mejor).fit(plano(tr), Y(tr))
        pred = lambda S: deshacer(S, *reg.predict(plano(S)).T)
        info["alpha"] = mejor
    else:
        keras.utils.set_random_seed(semilla)
        modelo = construir_red(tipo, len(cols) + (2 if con_dif else 0), D["train"]["X"].shape[1], lam)
        parada = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True),
                  keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PACIENCIA // 4, min_lr=1e-5)]
        hist = modelo.fit(entradas(tr), objetivos(tr), validation_data=(entradas(D["val"]), objetivos(D["val"])),
                          epochs=EPOCAS, batch_size=BATCH, callbacks=parada, verbose=0)
        info["history"] = {k: [float(x) for x in v] for k, v in hist.history.items()}
        info["epochs"] = len(hist.history["loss"])
        info["best_epoch"] = int(np.argmin(hist.history["val_loss"])) + 1
        pred = lambda S: deshacer(S, *[p.ravel() for p in modelo.predict(entradas(S), verbose=0)])
        # Activation of the physics term on the training set (share of samples above the limit)
        _, pv_tr = pred(tr)
        vmax_tr = calcular_vmax_fisico_kmh(tr["R"], tr["th"], mu=mu)
        info["train_pred_above_vmax_pct"] = float(np.mean(pv_tr > vmax_tr) * 100)
        info["train_actual_above_vmax_pct"] = float(np.mean(tr["v"] > vmax_tr) * 100)
    info["seconds"] = time.time() - t0
    return {k: pred(D[k]) for k in ("test", "kitti")}, info

In [ ]:
# Experiment list: (experiment group, horizon, model specification).
TODAS = list(range(N_FEATURES))
IDX = {c: i for i, c in enumerate(FEATURE_COLS)}
PREV = [IDX[f"preview_{d}m"] for d in DISTANCIAS_PREVIEW]
BG = [IDX["peralte_deg"], IDX["pendiente_pct"]]
DIN = [IDX_VEL, IDX_HR]

def especificaciones():
    lista = []
    for h in HORIZONTES:
        for nombre, tipo, lam in [("Persistence", "Persistence", 0), ("Constant trend", "Trend", 0),
                                  ("Ridge", "Ridge", 0), ("MLP", "MLP", 0), ("LSTM", "LSTM", 0), ("GRU", "GRU", 0),
                                  ("CNN-LSTM", "CNN-LSTM", 0.0), ("CNN-LSTM + PIDL", "CNN-LSTM", LAMBDA_FISICA)]:
            lista.append(("main", h, {"name": nombre, "kind": tipo, "lam": lam, "mu": MU_FISICA}))
    h = HORIZONTE_PRINCIPAL
    ablaciones = [("Dynamics only", DIN), ("Geometry only", PREV + BG), ("No preview", BG + DIN),
                  ("No banking/grade", PREV + DIN)]
    ablaciones += [(f"Preview {d} m only", [IDX[f"preview_{d}m"]] + BG + DIN) for d in DISTANCIAS_PREVIEW]
    for nombre, cols in ablaciones:
        lista.append(("ablation", h, {"name": nombre, "kind": "CNN-LSTM", "cols": cols, "lam": 0.0}))
    for w in VENTANAS_ABLACION:
        lista.append(("ablation", h, {"name": f"History {w / FRECUENCIA_HZ:.1f} s", "kind": "CNN-LSTM",
                                      "lam": 0.0, "window": w}))
        lista.append(("ablation", h, {"name": f"Persistence (history {w / FRECUENCIA_HZ:.1f} s)",
                                      "kind": "Persistence", "window": w}))
    for mu in MU_SENSIBILIDAD:
        for lam in LAMBDA_SENSIBILIDAD:
            if mu == MU_FISICA and lam == LAMBDA_FISICA:
                continue                      # already trained in the main experiment
            lista.append(("physics", h, {"name": f"PIDL mu={mu} lambda={lam}", "kind": "CNN-LSTM", "lam": lam, "mu": mu}))
    return lista

ESPECS = especificaciones()
print(f"{len(ESPECS)} model fits per fold, {len(ESPECS) * N_FOLDS} in total.")

In [ ]:
# Run the 5 folds. Every fold is saved to Drive; finished folds are loaded instead of retrained.

def ejecutar_fold(k):
    ids_tr, ids_va, ids_te = particion_fold(k)
    cache = {}
    def conjuntos(h, w):
        if (h, w) not in cache:
            cache[(h, w)] = {"train": apilar(dfs_pandaset, ids_tr, h, w), "val": apilar(dfs_pandaset, ids_va, h, w),
                             "test": apilar(dfs_pandaset, ids_te, h, w), "kitti": apilar(dfs_kitti, ids_kitti, h, w)}
        return cache[(h, w)]
    muestras, info_fold = [], []
    for grupo, h, spec in ESPECS:
        w = spec.get("window", WINDOW_SIZE)
        D = conjuntos(h, w)
        preds, info = ejecutar_modelo(spec, D, h, SEMILLA + k)
        for ds, clave in [("PandaSet", "test"), ("KITTI", "kitti")]:
            S, (pg, pv) = D[clave], preds[clave]
            ug, uv = ultimos(S["X"])
            muestras.append(pd.DataFrame({
                "fold": k + 1, "group": grupo, "model": spec["name"], "horizon_s": h / FRECUENCIA_HZ,
                "window_s": w / FRECUENCIA_HZ, "dataset": ds, "seq": S["seq"],
                "g": S["g"], "v": S["v"], "pg": pg, "pv": pv, "ug": ug, "uv": uv, "R": S["R"], "th": S["th"]}))
        info.update({"fold": k + 1, "group": grupo, "model": spec["name"], "horizon_s": h / FRECUENCIA_HZ})
        info_fold.append(info)
        print(f"  fold {k + 1} | {grupo:8s} | h={h / FRECUENCIA_HZ:.1f} s | {spec['name']:32s} | "
              f"{info.get('epochs', '-')} epochs (best {info.get('best_epoch', '-')}), {info.get('seconds', 0):.0f} s")
    return pd.concat(muestras, ignore_index=True), info_fold

todas_muestras, todo_info = [], []
for k in range(N_FOLDS):
    ruta_fold = os.path.join(RUTA_RESULTADOS, f"fold_{k + 1}.pkl")
    if os.path.exists(ruta_fold):
        with open(ruta_fold, "rb") as f:
            m, inf = pickle.load(f)
        print(f"Fold {k + 1} loaded from Drive.")
    else:
        print(f"Fold {k + 1}/{N_FOLDS} ...")
        m, inf = ejecutar_fold(k)
        with open(ruta_fold, "wb") as f:
            pickle.dump((m, inf), f)
    todas_muestras.append(m)
    todo_info += inf
M = pd.concat(todas_muestras, ignore_index=True)
INFO = pd.DataFrame(todo_info)
print(f"Done: {len(M)} prediction rows.")

## 11. Main results: all models, three horizons

* **PandaSet**: metrics pooled over the 5 test folds (each sequence predicted exactly once by a model that never saw it), plus mean ± SD over folds.
* **Statistics** (PandaSet): per-sequence MSE of each model vs. persistence and vs. the proposed model; two-sided **Wilcoxon signed-rank** test over sequences (independent units), Holm correction within each table; **95% bootstrap CI** (2000 resamples of sequences) of the pooled relative MSE change.
* **KITTI**: each fold model is evaluated on all KITTI sequences (zero-shot); mean ± SD over the 5 fold models.

In [ ]:
# Main table, statistical comparison and per-fold variability.
pd.set_option("display.width", 250)
SALIDAS = {"heading rate": ("g", "pg"), "speed": ("v", "pv")}
B_BOOT = 2000

def metricas(df):
    out = {}
    for nombre, (y, p) in SALIDAS.items():
        out[f"MSE {nombre}"] = mean_squared_error(df[y], df[p])
        out[f"R2 {nombre}"] = r2_score(df[y], df[p])
    return pd.Series(out)

def mse_por_secuencia(df, salida):
    y, p = SALIDAS[salida]
    return df.assign(e=(df[y] - df[p]) ** 2).groupby("seq")["e"].agg(["sum", "count"])

def holm(p):
    p = np.asarray(p, float)
    orden = np.argsort(p)
    ajust = np.empty_like(p)
    acumulado = 0.0
    for r, i in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (len(p) - r) * p[i]))
        ajust[i] = acumulado
    return ajust

def comparar(dfA, dfB, salida, semilla=SEMILLA):
    # Relative pooled MSE change of A vs B (%), bootstrap CI over sequences, Wilcoxon over sequences.
    a, b = mse_por_secuencia(dfA, salida), mse_por_secuencia(dfB, salida)
    a, b = a.loc[b.index], b
    if b["sum"].sum() == 0:
        return {"change_pct": np.nan, "ci_low": np.nan, "ci_high": np.nan, "seq_won": "-", "p_wilcoxon": np.nan}
    cambio = (a["sum"].sum() / b["sum"].sum() - 1) * 100
    rng = np.random.default_rng(semilla)
    idx = rng.integers(0, len(a), size=(B_BOOT, len(a)))
    with np.errstate(divide="ignore", invalid="ignore"):
        boot = (a["sum"].values[idx].sum(1) / b["sum"].values[idx].sum(1) - 1) * 100
    boot = boot[np.isfinite(boot)]
    ma, mb = a["sum"] / a["count"], b["sum"] / b["count"]
    try:
        p = stats.wilcoxon(ma, mb).pvalue
    except ValueError:
        p = np.nan
    return {"change_pct": cambio, "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5),
            "seq_won": f"{int((ma < mb).sum())}/{len(ma)}", "p_wilcoxon": p}

principal = M[M["group"] == "main"]
MODELOS = ["Persistence", "Constant trend", "Ridge", "MLP", "LSTM", "GRU", "CNN-LSTM", "CNN-LSTM + PIDL"]

# (a) pooled PandaSet metrics and (b) mean +/- SD over folds, for both datasets
filas = []
for (h, ds, mod), df in principal.groupby(["horizon_s", "dataset", "model"]):
    por_fold = df.groupby("fold").apply(metricas)
    fila = {"Horizon (s)": h, "Dataset": ds, "Model": mod}
    for c in por_fold.columns:
        fila[f"{c} pooled"] = metricas(df)[c] if ds == "PandaSet" else np.nan
        fila[f"{c} mean"], fila[f"{c} sd"] = por_fold[c].mean(), por_fold[c].std()
    filas.append(fila)
tabla_main = pd.DataFrame(filas)
tabla_main["Model"] = pd.Categorical(tabla_main["Model"], MODELOS)
tabla_main = tabla_main.sort_values(["Dataset", "Horizon (s)", "Model"], ascending=[False, True, True])
guardar_tabla(tabla_main.round(5), "table_main_results")
display(tabla_main.round(4))

# (c) statistical comparison on PandaSet: every model vs persistence, and the proposed model vs every model
filas = []
for h in HORIZONTES:
    hs = h / FRECUENCIA_HZ
    sub = principal[(principal["horizon_s"] == hs) & (principal["dataset"] == "PandaSet")]
    for ref, candidatos in [("Persistence", MODELOS[1:]), ("CNN-LSTM + PIDL", MODELOS[:-1])]:
        for mod in candidatos:
            for salida in SALIDAS:
                A = sub[sub["model"] == (mod if ref == "Persistence" else ref)]
                Bm = sub[sub["model"] == (ref if ref == "Persistence" else mod)]
                r = comparar(A, Bm, salida)
                etiqueta = f"{mod} vs Persistence" if ref == "Persistence" else f"CNN-LSTM + PIDL vs {mod}"
                filas.append({"Horizon (s)": hs, "Comparison": etiqueta, "Reference": ref, "Output": salida, **r})
tabla_stats = pd.DataFrame(filas)
tabla_stats["p_holm"] = tabla_stats.groupby(["Horizon (s)", "Reference", "Output"])["p_wilcoxon"].transform(holm)
guardar_tabla(tabla_stats.round(5), "table_statistics_pandaset")
display(tabla_stats.round(4))

# (d) KITTI zero-shot per sequence (prediction averaged over the 5 fold models), proposed model vs persistence
filas = []
for h in HORIZONTES:
    hs = h / FRECUENCIA_HZ
    sub = principal[(principal["horizon_s"] == hs) & (principal["dataset"] == "KITTI")]
    for mod in MODELOS[1:]:
        for salida in SALIDAS:
            filas.append({"Horizon (s)": hs, "Model": mod, "Output": salida,
                          **comparar(sub[sub["model"] == mod], sub[sub["model"] == "Persistence"], salida)})
tabla_kitti = pd.DataFrame(filas)
guardar_tabla(tabla_kitti.round(5), "table_statistics_kitti")
display(tabla_kitti.round(3))
print("KITTI rows pool the 5 fold models; CIs and Wilcoxon tests run over KITTI sequences.")

## 12. Ablation study (1.0 s horizon)

All ablations use the CNN-LSTM without the physics term, so the reference is **CNN-LSTM (full inputs)**. Changes are relative pooled MSE (negative = lower error than the full model), with bootstrap 95% CIs and Wilcoxon tests over PandaSet sequences. History-length variants are compared with the persistence baseline built on the same windows.

In [ ]:
# Ablations of inputs and history length.
hs = HORIZONTE_PRINCIPAL / FRECUENCIA_HZ
P_main = M[(M["horizon_s"] == hs) & (M["dataset"] == "PandaSet")]
completo = P_main[(P_main["group"] == "main") & (P_main["model"] == "CNN-LSTM")]
pers = P_main[(P_main["group"] == "main") & (P_main["model"] == "Persistence")]

filas = []
nombres_abl = [s["name"] for g, _, s in ESPECS if g == "ablation" and s["kind"] != "Persistence"]
for nombre in ["CNN-LSTM (full inputs)"] + nombres_abl:
    df = completo if nombre == "CNN-LSTM (full inputs)" else P_main[(P_main["group"] == "ablation") & (P_main["model"] == nombre)]
    base_p = pers
    if nombre.startswith("History"):
        base_p = P_main[P_main["model"] == f"Persistence ({nombre.lower()})"]
    for salida in SALIDAS:
        fila = {"Variant": nombre, "Output": salida, "MSE pooled": metricas(df)[f"MSE {salida}"],
                "R2 pooled": metricas(df)[f"R2 {salida}"]}
        vp = comparar(df, base_p, salida)
        fila.update({"vs persistence (%)": vp["change_pct"], "vs persistence p": vp["p_wilcoxon"]})
        if nombre != "CNN-LSTM (full inputs)" and not nombre.startswith("History"):
            vf = comparar(df, completo, salida)
            fila.update({"vs full (%)": vf["change_pct"], "vs full CI low": vf["ci_low"],
                         "vs full CI high": vf["ci_high"], "vs full p": vf["p_wilcoxon"], "seq won": vf["seq_won"]})
        filas.append(fila)
tabla_abl = pd.DataFrame(filas)
guardar_tabla(tabla_abl.round(5), "table_ablation")
display(tabla_abl.round(4))

sub = tabla_abl.dropna(subset=["vs full (%)"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, salida, nombre in zip(axes, SALIDAS, ["Heading rate", "Speed"]):
    s = sub[sub["Output"] == salida]
    y = np.arange(len(s))
    err = [s["vs full (%)"] - s["vs full CI low"], s["vs full CI high"] - s["vs full (%)"]]
    ax.barh(y, s["vs full (%)"], xerr=err, capsize=3, color=np.where(s["vs full (%)"] > 0, "#c0504d", "#4f81bd"))
    ax.set_yticks(y, s["Variant"])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("MSE change vs. full-input CNN-LSTM (%)")
    ax.set_title(f"{nombre} (positive = removing inputs hurts)", fontsize=10.5)
axes[0].invert_yaxis()
fig.tight_layout()
guardar_figura(fig, "fig12_ablation")
plt.show()

## 13. Sensitivity of the physics-informed term (1.0 s horizon)

For each friction coefficient μ (dry 0.7 → icy 0.1) and weight λ, the table reports the accuracy, the share of **actual** future speeds above the physical limit (how often real driving violates the constraint), the share of **predicted** speeds above it (test), and the activation of the penalty on the training set. λ = 0 is the CNN-LSTM without the physics term evaluated against the same limit.

In [ ]:
# Physics term: mu x lambda sensitivity.
P_fis = M[(M["horizon_s"] == hs) & (M["dataset"] == "PandaSet")]
filas = []
for mu in MU_SENSIBILIDAD:
    for lam in [0.0] + LAMBDA_SENSIBILIDAD:
        if lam == 0.0:
            nombre, grupo = "CNN-LSTM", "main"
        elif mu == MU_FISICA and lam == LAMBDA_FISICA:
            nombre, grupo = "CNN-LSTM + PIDL", "main"
        else:
            nombre, grupo = f"PIDL mu={mu} lambda={lam}", "physics"
        df = P_fis[(P_fis["group"] == grupo) & (P_fis["model"] == nombre)]
        vmax = calcular_vmax_fisico_kmh(df["R"].values, df["th"].values, mu=mu)
        exceso = np.clip(df["pv"].values - vmax, 0, None)
        inf = INFO[(INFO["group"] == grupo) & (INFO["model"] == nombre) & (INFO["horizon_s"] == hs)]
        ref = P_fis[(P_fis["group"] == "main") & (P_fis["model"] == "CNN-LSTM")]
        cs = comparar(df, ref, "speed") if lam > 0 else {"change_pct": 0.0, "p_wilcoxon": np.nan}
        filas.append({"mu": mu, "lambda": lam, "MSE speed": metricas(df)["MSE speed"],
                      "MSE heading rate": metricas(df)["MSE heading rate"],
                      "speed MSE vs lambda=0 (%)": cs["change_pct"], "p": cs["p_wilcoxon"],
                      "actual above v_max (%)": np.mean(df["v"].values > vmax) * 100,
                      "predicted above v_max (%)": np.mean(exceso > 0) * 100,
                      "mean excess (km/h)": exceso[exceso > 0].mean() if (exceso > 0).any() else 0.0,
                      "train penalty active (%)": inf["train_pred_above_vmax_pct"].mean() if lam > 0 else np.nan})
tabla_fis = pd.DataFrame(filas)
guardar_tabla(tabla_fis.round(5), "table_physics_sensitivity")
display(tabla_fis.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for lam in [0.0] + LAMBDA_SENSIBILIDAD:
    s = tabla_fis[tabla_fis["lambda"] == lam]
    etiqueta = "no physics term" if lam == 0 else f"$\\lambda$ = {lam:g}"
    axes[0].plot(s["mu"], s["MSE speed"], "o-", label=etiqueta)
    axes[1].plot(s["mu"], s["predicted above v_max (%)"], "o-", label=etiqueta)
s0 = tabla_fis[tabla_fis["lambda"] == 0.0]
axes[1].plot(s0["mu"], s0["actual above v_max (%)"], "k--", label="actual future speed")
for lam in LAMBDA_SENSIBILIDAD:
    s = tabla_fis[tabla_fis["lambda"] == lam]
    axes[2].plot(s["mu"], s["train penalty active (%)"], "o-", label=f"$\\lambda$ = {lam:g}")
for ax, t in zip(axes, ["Speed MSE (test)", "Speed above $v_{max}$ (test, %)", "Penalty active (training samples, %)"]):
    ax.set_xlabel("Friction coefficient $\\mu$")
    ax.set_title(t, fontsize=10.5)
    ax.invert_xaxis()
    ax.legend(fontsize=8.5)
fig.tight_layout()
guardar_figura(fig, "fig13_physics_sensitivity")
plt.show()

# Distribution of the physical limit with the nominal mu, and road vs vehicle radius
todos = pd.concat(dfs_pandaset.values())
print("Road radius from LiDAR (m), percentiles 5/25/50/75/95:", np.percentile(todos["R_road_m"], [5, 25, 50, 75, 95]).round(1))
print("Vehicle radius v/omega (m), percentiles 5/25/50/75/95:", np.percentile(todos["R_veh_m"], [5, 25, 50, 75, 95]).round(1))
print("Spearman(R_road, R_veh):", round(stats.spearmanr(todos["R_road_m"], todos["R_veh_m"])[0], 3))

## 14. Curve-stratified performance (1.0 s horizon, PandaSet)

Test samples are grouped by the magnitude of the actual mean heading rate over the horizon: **straight** (< 1 deg/s), **gentle curve** (1–3 deg/s) and **curve** (≥ 3 deg/s). The thresholds are lower than typical turn rates because the PandaSet clips contain mostly gentle urban curves.

In [ ]:
# Errors per curvature stratum.
CORTES = [0, 1, 3, np.inf]
ESTRATOS = ["Straight (<1 deg/s)", "Gentle curve (1-3 deg/s)", "Curve (>=3 deg/s)"]
P_est = M[(M["group"] == "main") & (M["horizon_s"] == hs) & (M["dataset"] == "PandaSet")].copy()
P_est["stratum"] = pd.cut(np.abs(P_est["g"]) * FRECUENCIA_HZ, CORTES, labels=ESTRATOS, right=False)

filas = []
for est in ESTRATOS:
    S = P_est[P_est["stratum"] == est]
    base = S[S["model"] == "Persistence"]
    for mod in MODELOS:
        df = S[S["model"] == mod]
        if len(df) == 0:
            continue
        fila = {"Stratum": est, "Model": mod, "N samples": len(df), "N sequences": df["seq"].nunique()}
        for salida in SALIDAS:
            fila[f"MSE {salida}"] = metricas(df)[f"MSE {salida}"]
            if mod != "Persistence":
                c = comparar(df, base, salida)
                fila[f"{salida} vs persistence (%)"] = c["change_pct"]
                fila[f"{salida} CI low"], fila[f"{salida} CI high"] = c["ci_low"], c["ci_high"]
        filas.append(fila)
tabla_est = pd.DataFrame(filas)
guardar_tabla(tabla_est.round(5), "table_curve_strata")
display(tabla_est.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
mostrar = ["Ridge", "LSTM", "GRU", "CNN-LSTM", "CNN-LSTM + PIDL"]
ancho = 0.8 / len(mostrar)
for ax, salida, nombre in zip(axes, SALIDAS, ["Heading rate", "Speed"]):
    for j, mod in enumerate(mostrar):
        s = tabla_est[tabla_est["Model"] == mod].set_index("Stratum").reindex(ESTRATOS)
        x = np.arange(len(ESTRATOS)) + (j - (len(mostrar) - 1) / 2) * ancho
        err = [s[f"{salida} vs persistence (%)"] - s[f"{salida} CI low"], s[f"{salida} CI high"] - s[f"{salida} vs persistence (%)"]]
        ax.bar(x, s[f"{salida} vs persistence (%)"], ancho, yerr=err, capsize=2, label=mod)
    ax.set_xticks(np.arange(len(ESTRATOS)), [e.replace(" (", "\n(") for e in ESTRATOS])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("MSE change vs. persistence (%)")
    ax.set_title(f"{nombre} (negative = lower error)", fontsize=10.5)
axes[1].legend(fontsize=8.5)
fig.tight_layout()
guardar_figura(fig, "fig14_curve_strata")
plt.show()

## 15. Figures 6–9 of the article (updated)

In [ ]:
# Figure 6: training and validation loss of the proposed model (fold 1, 1.0 s horizon).
inf = INFO[(INFO["group"] == "main") & (INFO["model"] == "CNN-LSTM + PIDL") & (INFO["horizon_s"] == hs) & (INFO["fold"] == 1)].iloc[0]
hh = inf["history"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hh["loss"], label="Training loss")
ax.plot(hh["val_loss"], label="Validation loss")
mejor = int(np.argmin(hh["val_loss"]))
ax.axvline(mejor, color="gray", ls=":", label=f"Best epoch ({mejor + 1})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Total loss (standardized units)")
ax.set_title("CNN-LSTM + PIDL training (fold 1, 1.0 s horizon)")
ax.legend()
guardar_figura(fig, "fig6_training_loss")
plt.show()

# Figure 7: predicted vs actual (PandaSet pooled over the 5 test folds; KITTI with the fold-1 model).
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for fila, ds in enumerate(["PandaSet", "KITTI"]):
    df = principal[(principal["model"] == "CNN-LSTM + PIDL") & (principal["horizon_s"] == hs) & (principal["dataset"] == ds)]
    if ds == "KITTI":
        df = df[df["fold"] == 1]
    for col, (y, p, unidad, nombre) in enumerate([("g", "pg", "heading rate (deg/frame)", "heading rate"),
                                                   ("v", "pv", "speed (km/h)", "speed")]):
        ax = axes[fila, col]
        ax.scatter(df[y], df[p], s=6, alpha=0.4)
        lim = [min(df[y].min(), df[p].min()), max(df[y].max(), df[p].max())]
        ax.plot(lim, lim, "k--", lw=1, label="Ideal (y = x)")
        etiqueta = "PandaSet (5 test folds)" if ds == "PandaSet" else "KITTI Road (zero-shot, fold-1 model)"
        ax.set_title(f"{etiqueta} - {nombre}\n$R^2$ = {r2_score(df[y], df[p]):.3f}")
        ax.set_xlabel(f"Actual {unidad}"); ax.set_ylabel(f"Predicted {unidad}")
        ax.legend(loc="upper left")
fig.tight_layout()
guardar_figura(fig, "fig7_pred_vs_actual")
plt.show()

# Figure 8: speed along the first KITTI sequence (fold-1 model).
primera = ids_kitti[0]
sel = lambda mod: principal[(principal["model"] == mod) & (principal["horizon_s"] == hs) & (principal["dataset"] == "KITTI")
                            & (principal["fold"] == 1) & (principal["seq"] == primera)]
d1, dp = sel("CNN-LSTM + PIDL"), sel("Persistence")
vmax1 = calcular_vmax_fisico_kmh(d1["R"].values, d1["th"].values)
t = np.arange(len(d1)) / FRECUENCIA_HZ
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, d1["v"].values, label="Actual mean speed over the next 1.0 s (OXTS)")
ax.plot(t, d1["pv"].values, label="CNN-LSTM + PIDL")
ax.plot(t, dp["pv"].values, ":", label="Persistence baseline")
y_sup = 1.3 * max(d1["v"].max(), d1["pv"].max())
ax.plot(t, np.where(vmax1 < y_sup, vmax1, np.nan), "--", label="Physical $v_{max}$ (LiDAR road geometry, $\\mu$ = 0.7)")
ax.set_ylim(0, y_sup)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Speed (km/h)")
ax.set_title(f"KITTI {primera} - zero-shot speed prediction")
ax.legend(loc="lower right", fontsize=9)
guardar_figura(fig, "fig8_kitti_speed")
plt.show()

# Figure 9: relative MSE change vs persistence with bootstrap 95% CIs (PandaSet) and fold SD (KITTI).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, salida, nombre in zip(axes, SALIDAS, ["Heading rate", "Speed"]):
    s = tabla_stats[(tabla_stats["Comparison"] == "CNN-LSTM + PIDL vs Persistence") & (tabla_stats["Output"] == salida)]
    ax.errorbar(s["Horizon (s)"] - 0.04, s["change_pct"], yerr=[s["change_pct"] - s["ci_low"], s["ci_high"] - s["change_pct"]],
                fmt="-o", capsize=3, label="PandaSet (5 test folds, 95% CI)")
    for x, y, w in zip(s["Horizon (s)"] - 0.04, s["change_pct"], s["seq_won"]):
        ax.annotate(w, (x, y), textcoords="offset points", xytext=(8, -3), fontsize=8.5)
    k_ = tabla_kitti[(tabla_kitti["Model"] == "CNN-LSTM + PIDL") & (tabla_kitti["Output"] == salida)]
    ax.errorbar(k_["Horizon (s)"] + 0.04, k_["change_pct"], yerr=[k_["change_pct"] - k_["ci_low"], k_["ci_high"] - k_["change_pct"]],
                fmt="--s", capsize=3, label="KITTI (zero-shot, 95% CI)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks([h / FRECUENCIA_HZ for h in HORIZONTES])
    ax.set_xlabel("Prediction horizon (s)"); ax.set_ylabel("MSE change vs. persistence (%)")
    ax.set_title(f"{nombre} (negative = lower error than persistence)", fontsize=10.5)
axes[1].legend(loc="upper left", fontsize=9)
fig.tight_layout()
guardar_figura(fig, "fig9_mse_vs_persistence")
plt.show()

## 16. Download

Download `revision_results.zip` and send it back: it contains all figures (PDF + PNG) and all tables (CSV).

In [ ]:
# Pack all figures and tables; also copy the zip to Drive.
import shutil
INFO.drop(columns=["history"], errors="ignore").to_csv(os.path.join(FIG_DIR, "training_info.csv"), index=False)
shutil.make_archive("revision_results", "zip", FIG_DIR)
shutil.copy("revision_results.zip", RUTA_RESULTADOS)
from google.colab import files
files.download("revision_results.zip")